In [1]:
%reset -f

In [2]:
import time

running_time = time.time()

In [3]:
!sudo apt update > /dev/null; sudo apt install -y gcc > /dev/null

In [4]:
!pip install torch > /dev/null

In [5]:
!pip install torch_geometric > /dev/null

In [6]:
!pip install pytorch_lightning > /dev/null

In [7]:
!pip install torchmetrics > /dev/null

In [8]:
!pip install -r ./../r.txt > /dev/null

## implementing the exemple in (https://graphein.ai/notebooks/pscdb_baselines.html)

In [1]:
import sys
sys.path.append('../')

import logging
logging.getLogger('matplotlib').setLevel(logging.CRITICAL)
logging.getLogger('graphein').setLevel(logging.INFO)

In [2]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import pytorch_lightning as pl
from tqdm import tqdm
import networkx as nx
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")


In [3]:
df = pd.read_csv("../data/structural_rearrangement_data.csv")
pdbs = df["Free PDB"]
y = [torch.argmax(torch.Tensor(lab)).type(torch.LongTensor) for lab in LabelBinarizer().fit_transform(df.motion_type)]

In [4]:
import Builder
# from PDBGraphStore import PDBGraphStore
from PDBGraphStore import PDBGraphStore
import pickle as pk

In [5]:
pdb_store = PDBGraphStore(None)

# pdb_graph_store.insert({"1BXL": g})

In [6]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import add_hydrogen_bond_interactions, add_peptide_bonds, add_k_nn_edges
from graphein.protein.graphs import construct_graph
from graphein.protein.utils import download_pdb
import os

from functools import partial

# Override config with constructors
constructors = {
    "edge_construction_functions": [partial(add_k_nn_edges, k=3, long_interaction_threshold=0)],
    #"edge_construction_functions": [add_hydrogen_bond_interactions, add_peptide_bonds],
    #"node_metadata_functions": [add_dssp_feature]
}

config = ProteinGraphConfig(**constructors)
print(config.dict())

# Make graphs
y_list = []

pdb_data_path = "../data"

for idx, pdb in enumerate(tqdm(pdbs)):
    if os.path.exists(f'{pdb_data_path}/{pdb}.pdb'):
        pdb_file = os.path.abspath(f'{pdb_data_path}/{pdb}.pdb')
    else:
        try:
            pdb_file = download_pdb(pdb, f'{pdb_data_path}/')
        except Exception as e:
            print(str(idx) + "processing error....")
            pass
    
    if pdb_file == None:
        print(str(idx) + "processing error....")
        pass
    
    try:
        graph = construct_graph(config=config, path=pdb_file)
        g_config = graph.graph["config"]
        g_pdb_code = graph.graph["pdb_code"]
        graph.graph.clear()
        graph.graph['config'] = g_config
        graph.graph['pdb_code'] = g_pdb_code

        pdb_store.insert_pdb({pdb: [graph]})

        y_list.append(y[idx])
    except Exception as e:
        print(str(idx) + ' processing error...')
        print(e)
        pass

del graph
    

{'granularity': 'CA', 'keep_hets': [], 'insertions': True, 'alt_locs': 'max_occupancy', 'pdb_dir': None, 'verbose': False, 'exclude_waters': True, 'deprotonate': False, 'protein_df_processing_functions': None, 'edge_construction_functions': [functools.partial(<function add_k_nn_edges at 0x7f84ea718a40>, k=3, long_interaction_threshold=0)], 'node_metadata_functions': [<function meiler_embedding at 0x7f84ea71ad40>], 'edge_metadata_functions': None, 'graph_metadata_functions': None, 'get_contacts_config': None, 'dssp_config': None}


  0%|          | 0/891 [00:00<?, ?it/s]

Output()

  0%|          | 1/891 [00:00<03:02,  4.89it/s]

Output()

...

Output()

100%|█████████▉| 889/891 [02:41<00:00,  8.83it/s]

Output()

Output()

100%|██████████| 891/891 [02:41<00:00,  5.50it/s]


In [7]:
# from pympler.asizeof import asizeof
# from MemoryMeasuring import MemoryMeasuring

# memory = MemoryMeasuring(pdb_store)

# print(f'pdb_store object memory (MB): {asizeof(pdb_store)/1024/1024}')
# print(f'sum of body_parts objects (MB){memory.total_memory()}')

# print(f'node_attr_key_value_mapping memory (MB): {memory.node_global_attr_keyvalue_mapping_memory()}')
# print(f'node_attr_values memory (MB): {memory.node_attr_values_memory()}')

In [25]:
print(list(pdb_store.get_body_parts()['node_attr_values'].keys())[:10])

[(-23.0, -12.95, -39.107), 42.790000915527344, (-20.79, -13.271, -42.173), 36.7599983215332, (-18.083, -15.647, -43.368), 31.809999465942383, (-14.54, -14.628, -44.291), 31.489999771118164, (-15.438, -15.327, -47.928), 31.8700008392334]


In [11]:
p0m = pdb_store.extract_pdb('2p0m')

for n in p0m.nodes():
    print(p0m.nodes[n])
    break

{'chain_id': 'A', 'residue_name': 'VAL', 'residue_number': '3', 'atom_type': 'CA', 'element_symbol': 'C', 'meiler': dim_1    3.67
dim_2    0.14
dim_3    3.00
dim_4    1.22
dim_5    6.02
dim_6    0.27
dim_7    0.49
Name: VAL, dtype: float64, 'coords': array([  9.892, 118.76 ,  35.686], dtype=float32), 'b_factor': 53.290000915527344}


In [7]:
print(pdb_store)

PDBGraphStore with 788 pdbs


In [8]:
from graphein.ml.conversion import GraphFormatConvertor

format_convertor = GraphFormatConvertor('nx', 'pyg',
                                        verbose = 'gnn',
                                        columns = None)

In [9]:
print(len(pdbs))

891


In [10]:
pyg_list = []

In [11]:
for pdb in tqdm(pdbs):
    g = pdb_store.extract_pdb(pdb)
    pyg_list.append(format_convertor(g))

del g

100%|██████████| 891/891 [01:03<00:00, 13.98it/s]


In [12]:
for idx, g in enumerate(pyg_list):
    g.y = y_list[idx]
    g.coords = torch.FloatTensor(g.coords)

In [13]:
pyg_list = [
    g for g in pyg_list
    if g.coords.shape[0] == len(g.node_id)
]

In [14]:
config_default = dict(
    n_hid = 8,
    n_out = 8,
    batch_size = 4,
    dropout = 0.5,
    lr = 0.001,
    num_heads = 32,
    num_att_dim = 64,
    model_name = 'GCN'
)

class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)

config = Struct(**config_default)

global model_name
model_name = config.model_name

In [15]:
import numpy as np
np.random.seed(42)
idx_all = np.arange(len(pyg_list))
np.random.shuffle(idx_all)

train_idx, valid_idx, test_idx = np.split(idx_all, [int(.8*len(pyg_list)), int(.9*len(pyg_list))])
train, valid, test = [pyg_list[i] for i in train_idx], [pyg_list[i] for i in valid_idx], [pyg_list[i] for i in test_idx]

from torch_geometric.data import DataLoader
train_loader = DataLoader(train, batch_size=config.batch_size, shuffle = True, drop_last = True)
valid_loader = DataLoader(valid, batch_size=32)
test_loader = DataLoader(test, batch_size=32)

In [16]:
pyg_list[0]

Data(edge_index=[2, 2210], node_id=[635], coords=[635, 3], num_nodes=635, y=1)

In [17]:
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_add_pool
from torch.nn.functional import mse_loss, nll_loss, relu, softmax, cross_entropy
from torch.nn import functional as F
from torchmetrics.functional import accuracy

In [18]:
class GraphNets(pl.LightningModule):
    def __init__(self):
        super().__init__()

        if model_name == 'GCN':
            self.layer1 = GCNConv(in_channels=3, out_channels=config.n_hid)
            self.layer2 = GCNConv(in_channels=config.n_hid, out_channels=config.n_out)

        elif model_name == 'GAT':
            self.layer1 = GATConv(3, config.num_att_dim, heads=config.num_heads, dropout=config.dropout)
            self.layer2 = GATConv(config.num_att_dim * config.num_heads, out_channels = config.n_out, heads=1, concat=False,
                                 dropout=config.dropout)

        elif model_name == 'GraphSAGE':
            self.layer1 = SAGEConv(3, config.n_hid)
            self.layer2 = SAGEConv(config.n_hid, config.n_out)

        self.decoder = nn.Linear(config.n_out, 7)

    def forward(self, g):
        x = g.coords
        x = F.dropout(x, p=config.dropout, training=self.training)
        x = F.elu(self.layer1(x, g.edge_index))
        x = F.dropout(x, p=config.dropout, training=self.training)
        x = self.layer2(x, g.edge_index)
        x = global_add_pool(x, batch=g.batch)
        x = self.decoder(x)
        return softmax(x)

    def training_step(self, batch, batch_idx):
        x = batch
        y = x.y
        y_hat = self(x)
        loss = cross_entropy(y_hat, y)
        acc = accuracy(
            y_hat,
            y,
            task="multiclass",
            num_classes=7
        )

        self.log("train_loss", loss)
        self.log("train_acc", acc)
        return loss

    def validation_step(self, batch, batch_idx):
        x = batch
        y = x.y
        y_hat = self(x)
        loss = cross_entropy(y_hat, y)
        acc = accuracy(
            y_hat,
            y,
            task="multiclass",
            num_classes=7
        )
        self.log("valid_loss", loss)
        self.log("valid_acc", acc)

    def test_step(self, batch, batch_idx):
        x = batch
        y = x.y
        y_hat = self(x)
        loss = cross_entropy(y_hat, y)
        acc = accuracy(
            y_hat,
            y,
            task="multiclass",
            num_classes=7
        )

        y_pred_softmax = torch.log_softmax(y_hat, dim = 1)
        y_pred_tags = torch.argmax(y_pred_softmax, dim = 1)
        f1 = f1_score(y.detach().cpu().numpy(), y_pred_tags.detach().cpu().numpy(), average = 'weighted')

        self.log("test_loss", loss)
        self.log("test_acc", acc)
        self.log("test_f1", f1)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=config.lr)
        return optimizer

In [19]:
GraphNets()

GraphNets(
  (layer1): GCNConv(3, 8)
  (layer2): GCNConv(8, 8)
  (decoder): Linear(in_features=8, out_features=7, bias=True)
)

In [20]:
from pytorch_lightning.callbacks import ModelCheckpoint
import os

file_path = './graphein_model'
if not os.path.exists(file_path):
    os.mkdir(file_path)

checkpoint_callback = ModelCheckpoint(
    monitor="valid_loss",
    dirpath=file_path,
    filename="model-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
    mode="min",
)

In [21]:
model = GraphNets()
trainer = pl.Trainer(max_epochs=200, accelerator='auto', callbacks=[checkpoint_callback])
trainer.fit(model, train_loader, valid_loader)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer1  │ GCNConv │     32 │ train │     0 │
│ 1 │ layer2  │ GCNConv │     72 │ train │     0 │
│ 2 │ decoder │ Linear  │     63 │ train │     0 │
└───┴─────────┴─────────┴────────┴───────┴───────┘

Trainable params: 167                                                                                              
Non-trainable params: 0                                                                                            
Total params: 167                                                                                                  
Total estimated model params size (MB): 0.001                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=200` reached.


In [22]:
best_model = GraphNets.load_from_checkpoint(checkpoint_callback.best_model_path)
out_best_test = trainer.test(best_model, test_loader)[0]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.31650641560554504    │
│          test_f1          │    0.15863344073295593    │
│         test_loss         │     1.848915696144104     │
└───────────────────────────┴───────────────────────────┘

In [31]:
with open("./time_graphstore.txt", 'a') as f:
    f.write(f'\n{str(time.time() - running_time)}')

In [32]:
%reset -f